# Memory Context Provider

This notebook introduces the [`neo4j-agent-memory`](https://github.com/neo4j-labs/agent-memory) package — a graph-native memory system for AI agents. Unlike the `agent-framework-neo4j` provider in Lab 6 that retrieves from a static knowledge base, agent memory is dynamic: it grows with every conversation as the agent learns user preferences, extracts entities, and records its own reasoning.

You'll configure this memory system as a **context provider** that runs automatically during agent conversations.

## What you will learn

- How to configure `Neo4jMicrosoftMemory` as a MAF context provider
- How the three memory types work together (short-term, long-term, reasoning)
- How entity extraction runs automatically after every agent turn
- How memory context is injected before each agent invocation

## Memory Types

The provider supports three memory types, each with different retrieval mechanics:

- **Short-term** — Recent conversation messages retrieved by semantic search. Provides conversational continuity.
- **Long-term** — Entities, preferences, and facts extracted from conversations. Stored as graph nodes in Neo4j.
- **Reasoning** — Traces of past agent task executions. Enables learning from experience.

***

Load the environment variables and import the required modules.

In [ ]:
import sys
sys.path.insert(0, '../shared')

import asyncio

from pydantic import SecretStr

from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

from neo4j_agent_memory import MemoryClient, MemorySettings
from neo4j_agent_memory.integrations.microsoft_agent import (
    Neo4jContextProvider,
    Neo4jMicrosoftMemory,
)

from azure_embedder import get_memory_embedder
from config import get_agent_config, Neo4jConfig

## Configure Memory Settings

The `MemorySettings` configures the connection to Neo4j and the embedding provider. We'll use the same Neo4j instance and Azure AI endpoint from the workshop environment.

In [ ]:
# Get workshop configuration
config = get_agent_config()
neo4j_config = Neo4jConfig()

# Configure memory settings
settings = MemorySettings(
    neo4j={
        "uri": neo4j_config.uri,
        "username": neo4j_config.username,
        "password": SecretStr(neo4j_config.password),
    },
    extraction={
        "enable_gliner": False,  # GLiNER disabled: downloads ~500MB model, impractical in a workshop
    },
)

print(f"Neo4j URI: {neo4j_config.uri}")
print(f"Memory settings configured")

## Create the Memory Client and Unified Memory

The `Neo4jMicrosoftMemory` class provides a unified interface that combines:
- A `Neo4jContextProvider` (automatically injects context before agent runs)
- A `Neo4jChatMessageStore` (persists conversation history)
- Direct access to all three memory types

In [ ]:
# Create memory client
memory_client = MemoryClient(settings, embedder=get_memory_embedder())
await memory_client.__aenter__()

# Create unified memory interface
session_id = "workshop-demo"

memory = Neo4jMicrosoftMemory.from_memory_client(
    memory_client=memory_client,
    session_id=session_id,
    include_short_term=True,
    include_long_term=True,
    include_reasoning=True,
    extract_entities=True,
)

print(f"Session: {session_id}")
print(f"Memory created with context provider and chat store")

> The `Neo4jMicrosoftMemory` creates both a context provider (`memory.context_provider`) and a chat message store (`memory.chat_store`). The context provider implements `BaseContextProvider` and runs automatically.
>
> Entity extraction runs **after every agent turn** via the `after_run()` hook. With `extract_entities=True`, the pipeline processes each response and stores discovered entities as graph nodes in Neo4j.

***

## Run a Multi-Turn Conversation

Attach the memory context provider to the agent. The provider will:
1. **Before each run**: Retrieve relevant conversation history, entities, and preferences
2. **After each run**: Save messages and extract entities from the conversation

Watch how the memory lifecycle plays out across three turns:

1. **Turn 1** — The user states an interest. `after_run()` saves the message and extracts entities (e.g., "Apple" as an ORGANIZATION).
2. **Turn 2** — The user asks a follow-up. `before_run()` retrieves the previous message and extracted entities, injecting them into context.
3. **Turn 3** — The user asks the agent to recall. The agent now has both conversation history and extracted entities available.

In [ ]:
async def run_conversation():
    async with AzureCliCredential() as credential:
        async with AzureAIClient(
            project_endpoint=config.project_endpoint,
            model_deployment_name=config.model_name,
            credential=credential,
        ) as client:
            agent = client.as_agent(
                name="workshop-memory-agent",
                instructions=(
                    "You are a helpful assistant with persistent memory. "
                    "You can remember previous conversations and user preferences. "
                    "When you notice the user expressing a preference, acknowledge it."
                ),
                context_providers=[memory.context_provider],
            )
            session = agent.create_session()

            # First interaction
            queries = [
                "Hi! I'm interested in learning about Apple's products.",
                "What about their risk factors? I'm particularly concerned about supply chain issues.",
                "Can you remind me what we discussed about Apple?",
            ]

            for query in queries:
                print(f"User: {query}\n")
                print("Assistant: ", end="", flush=True)

                response = await agent.run(query, session=session)
                print(response.text)
                print("\n" + "-" * 50 + "\n")

    await asyncio.sleep(0.1)

await run_conversation()

> Notice how the agent's responses become more contextually aware as the conversation progresses. The memory context provider injects relevant conversation history before each response.

***

## Explore Memory Contents

Let's look at what the memory system has stored.

In [ ]:
# Check what's in memory
results = await memory.search_memory(
    query="Apple products and risks",
    include_messages=True,
    include_entities=True,
    include_preferences=True,
    limit=5,
)

print("=== Memory Search Results ===\n")

if results.get("messages"):
    print(f"Messages found: {len(results['messages'])}")
    for msg in results["messages"][:3]:
        print(f"  [{msg['role']}] {msg['content'][:100]}...")
    print()

if results.get("entities"):
    print(f"Entities found: {len(results['entities'])}")
    for entity in results["entities"][:5]:
        print(f"  {entity['name']} ({entity['type']}): {entity.get('description', 'N/A')[:80]}")
    print()

if results.get("preferences"):
    print(f"Preferences found: {len(results['preferences'])}")
    for pref in results["preferences"][:3]:
        print(f"  [{pref['category']}] {pref['preference']}")

The memory context provider handles **passive** context injection — it runs automatically before and after every agent turn. In the next notebook, you'll explore the **entity extraction pipeline** in detail — the mechanism that turns unstructured conversation text into structured graph nodes. After that, you'll add **active** memory tools that give the agent explicit control over its memory.

***

[View the complete code](../financial_data_load/solution_srcs/07_01_memory_context_provider.py)

[Continue to the Entity Extraction Notebook](02_entity_extraction.ipynb)

In [ ]:
# Cleanup
await memory_client.__aexit__(None, None, None)